# Script de coleta e exportação
Este script realiza a coleta dos dados de interesse para análise.  
Ao final de sua execução total devem ser gerados 6 arquivos disponíveis para download:  
* 01 cd referente à taxa de pessoas desocupadas (trimestral, desde 2012 até atual);
* 01 cd referente à taxa de inadimplencia (mensal desde 2011 até atual);  
* 01 cd referente à taxa de variação do IPCA (Mensal desde 2014 até atual);  
* 01 cd referente à taxa de variação do PIB (trimestral, desde 1996 até atual);  
* 02 cd referentes à todas as as taxas juntas, por trimestre desde o ano de 2015 até  atual. Formatos xlsx e csv.

In [1]:
import pandas as pd
import requests, zipfile, io

# Download nº 1:
**Nome:** Inadimplência da carteira de crédito - Total  
**Descrição:** Série histórica das taxas de inadimplência mensais no Brasil para pessoas físicas e jurídicas desde 2011.  
**Fonte:** Banco Central dos Brasil

In [2]:
url_bcb = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.21082/dados?formato=csv"
df_inadimplencia = pd.read_csv(url_bcb, sep=';')

In [3]:
df_inadimplencia['valor'] = df_inadimplencia['valor'].str.replace(',', '.').astype(float)

In [6]:
df_inadimplencia['data'] = pd.to_datetime(df_inadimplencia['data'], format='%d/%m/%Y')

In [8]:
df_inadimplencia = df_inadimplencia.rename(columns={'valor': 'txa_inadimplencia'})

In [25]:
df_inadimplencia.head(3)

,data,txa_inadimplencia,trimestre
0,2011-03-01,3.17,2011Q1
1,2011-04-01,3.24,2011Q2
2,2011-05-01,3.37,2011Q2


Exportar df_inadimplencia MENSAL:

In [16]:
# Exportação do df_inadimplencia em formato mensal
df_inadimplencia.to_csv("df_inadimplencia_mensal.csv", index=False)

A partir desta célula o df passará a ser trimestral, para fins de análises conjuntas com outras variáveis que usam apenas essa configuração de tempo.

In [17]:
df_inadimplencia['trimestre'] = df_inadimplencia['data'].dt.to_period('Q')

In [18]:
# Média dos dados mensal para trimestral
df_inadimplencia_trimestral = (
    df_inadimplencia
    .groupby('trimestre', as_index=False)
    .agg(media_inadimplencia=('txa_inadimplencia', 'mean'))
)

df_inadimplencia_trimestral['trimestre'] = df_inadimplencia_trimestral['trimestre'].astype(str)

In [19]:
df_inadimplencia_trimestral.head(3)

,trimestre,media_inadimplencia
0,2011Q1,3.170000
1,2011Q2,3.310000
2,2011Q3,3.443333


# Download 2
**Nome:** Tabela 6691 - IPCA - Série histórica com número-índice, variação mensal e variações acumuladas a partir de 12 meses (a partir de novembro/2014)

**Parâmetros:**  
* Variável = IPCA - Variação mensal (%)  
* Mês = todas as caixas de seleção  
* Unidade Territorial = Brasil

**Fonte:** https://sidra.ibge.gov.br/Tabela/6691

In [20]:
url_ipca = "https://apisidra.ibge.gov.br/values/t/6691/n1/all/v/63/p/all/d/v63%202"
df_ipca = pd.read_json(url_ipca)

In [21]:
# Seleção das colunas de interesse
df_ipca = df_ipca[['V','D3C']]

In [22]:
# Ajuste do cabeçalho
df_ipca.columns = df_ipca.iloc[0]
df_ipca = df_ipca.drop(0).reset_index(drop=True)

In [23]:
# Converter coluna 'Valor' para float
df_ipca['Valor'] = df_ipca['Valor'].astype(float)
df_ipca = df_ipca.rename(columns={'Valor': 'txa_variacao_ipca'})
# E coluna 'Mês (Código)' para datetime
df_ipca['Mês (Código)'] = pd.to_datetime(df_ipca['Mês (Código)'], format='%Y%m')

In [24]:
df_ipca.head()

,txa_variacao_ipca,Mês (Código)
0,0.51,2014-11-01
1,0.78,2014-12-01
2,1.24,2015-01-01
3,1.22,2015-02-01
4,1.32,2015-03-01


Exportar df_ipca MENSAL:

In [26]:
# Exportação do df_ipca em formato mensal
df_ipca.to_csv("df_ipca_mensal.csv", index=False)

A partir desta célula o df passará a ser trimestral, para fins de análises conjuntas com outras variáveis que usam apenas essa configuração de tempo.

In [27]:
# Criação da coluna trimestre
df_ipca['trimestre'] = df_ipca['Mês (Código)'].dt.to_period('Q')

# Média dos dados mensal para trimestral
df_ipca_trimestral = (
    df_ipca
    .groupby('trimestre', as_index=False)
    .agg(media_ipca=('txa_variacao_ipca', 'mean'))
)

In [28]:
df_ipca_trimestral['trimestre'] = df_ipca_trimestral['trimestre'].astype(str)

In [53]:
df_ipca_trimestral.head(3)

,trimestre,media_ipca
0,2014Q4,0.645000
1,2015Q1,1.260000
2,2015Q2,0.746667


# Download 3:  
**Nome:** Tabela 4093 - Pessoas de 14 anos ou mais de idade, total, na força de trabalho, ocupadas, desocupadas, fora da força de trabalho, em situação de informalidade e respectivas taxas e níveis, por sexo  
**Parâmetros:**  
* Variáveis: Pessoas de 14 anos ou mais de idade, na força de trabalho, na semana de referência (Mil pessoas), Pessoas de 14 anos ou mais de idade ocupadas na semana de referência (Mil pessoas) e Pessoas de 14 anos ou mais de idade, desocupadas na semana de referência (Mil pessoas).  
* Sexo = Total  
* Trimestre = todas as caixas de seleção  
* Unidade Territorial = Brasil  

**Descrição:** Quantidade de pessoas na força de trabalho, ocupadas e desocupadas.  

**Fonte:** IBGE - Pesquisa Nacional por Amostra de Domicílios Contínua trimestral (https://sidra.ibge.gov.br/tabela/4093)

In [30]:
# Download
url_forcaDeTrabalho = "https://apisidra.ibge.gov.br/values/t/4093/n1/all/v/4088,4090,4092/p/all/c2/6794"
df_ft = pd.read_json(url_forcaDeTrabalho)

In [31]:
# Seleção das colunas de interesse
df_ft = df_ft[['V', 'D2C', 'D3C']]

In [32]:
# Ajuste do cabeçalho
df_ft.columns = df_ft.iloc[0]
df_ft = df_ft.drop(0).reset_index(drop=True)

In [33]:
# Converter 'Valor' para inteiro
df_ft['Valor'] = df_ft['Valor'].astype(int)

# Substituir os códigos numéricos por descrições intuitivas
mapa_variaveis = {
    '4088': 'Forca de Trabalho',      # Força de Trabalho
    '4090': 'Ocupados',               # Ocupados
    '4092': 'Desocupados'             # Desocupados
}

df_ft['Variável (Código)'] = df_ft['Variável (Código)'].map(mapa_variaveis)

In [34]:
# Calcular percentual de desocupados

# pivotar para ter cada variável como coluna:
df_desocupados = df_ft.pivot_table(
    index='Trimestre (Código)',
    columns='Variável (Código)',
    values='Valor',
    aggfunc='sum'
).reset_index()

# criar a coluna com o cálculo de percentual de desocupados
df_desocupados['taxa_desocupados'] = (df_desocupados['Desocupados'] / df_desocupados['Forca de Trabalho']) * 100
df_desocupados['taxa_desocupados'] = df_desocupados['taxa_desocupados'].round(2)

In [35]:
# Padronização da escrita do trimestre
df_desocupados['Trimestre (Código)'] = (
    df_desocupados['Trimestre (Código)'].astype(str)
    .str[:4] + 'Q' +
    df_desocupados['Trimestre (Código)'].astype(str).str[-2:].astype(int).astype(str)
)

In [36]:
df_desocupados = df_desocupados.rename(columns={'Trimestre (Código)': 'trimestre'})

In [37]:
df_desocupados.drop(['Desocupados', 'Forca de Trabalho', 'Ocupados'], axis=1, inplace=True)

In [54]:
df_desocupados.head(3)

Variável (Código),trimestre,taxa_desocupados
0,2012Q1,7.99
1,2012Q2,7.57
2,2012Q3,7.12


In [39]:
# Exportação do df
df_desocupados.to_csv("df_desocupados_trimestral.csv", index=False)

# Download 4:
**Nome:** Tabela 5932 - Taxa de variação do índice de volume trimestral  
**Descrição:**  
* Variável = Taxa acumulada em quatro trimestres (em relação ao mesmo período do ano anterior) (%): 1 de 1 casas decimais
* Setores e subsetores = PIB a preços de mercado
* Trimestre = Todas as caixas de seleção  
* Unidade Territorial = Brasil  

**Fonte:** https://sidra.ibge.gov.br/Tabela/5932

In [40]:
url_pib = "https://apisidra.ibge.gov.br/values/t/5932/n1/all/v/6562/p/all/c11255/90707/d/v6562%201"
df_pib = pd.read_json(url_pib)

In [41]:
# Manter apenas colunas relevantes
df_pib = df_pib[['D3C', 'V']]

In [42]:
# Ajuste do cabeçalho
df_pib.columns = df_pib.iloc[0]
df_pib = df_pib.drop(0).reset_index(drop=True)

In [43]:
# Converter coluna 'Valor' para float
df_pib['Valor'] = df_pib['Valor'].astype(float)
df_pib = df_pib.rename(columns={'Valor': 'txa_variacao_pib'})

In [44]:
# Padronização das colunas trimestrais com os dfs anteriores
df_pib['Trimestre (Código)'] = (
    df_pib['Trimestre (Código)'].astype(str)
    .str[:4] + 'Q' +
    df_pib['Trimestre (Código)'].astype(str).str[-2:].astype(int).astype(str)
)

In [45]:
df_pib.rename(columns={'Trimestre (Código)': 'trimestre'}, inplace=True)

In [55]:
df_pib.head(3)

,trimestre,txa_variacao_pib
0,1996Q1,2.5
1,1996Q2,2.1
2,1996Q3,2.3


In [47]:
# Exportação do df
df_pib.to_csv("df_pib_trimestral.csv", index=False)

# União dos conjuntos de dados

In [48]:
cd_final_trimestral = df_pib \
    .merge(df_inadimplencia_trimestral, on='trimestre', how='inner') \
    .merge(df_desocupados, on='trimestre', how='inner') \
    .merge(df_ipca_trimestral, on='trimestre', how='inner')

In [49]:
cd_final_trimestral.head()

,trimestre,txa_variacao_pib,media_inadimplencia,taxa_desocupados,media_ipca
0,2014Q4,0.5,2.846667,6.56,0.645000
1,2015Q1,-0.7,2.830000,8.02,1.260000
2,2015Q2,-1.3,2.966667,8.41,0.746667
3,2015Q3,-2.2,3.090000,9.00,0.460000
4,2015Q4,-3.5,3.330000,9.06,0.930000


# Exportar df_final

In [51]:
cd_final_trimestral.to_excel("Conjunto Final Unificado - Trimestral.xlsx", index=False)

In [52]:
cd_final_trimestral.to_csv("Conjunto Final Unificado - Trimestral.csv", index=False)